# 00 · Welcome to PySpark

This course assumes **zero** Spark knowledge. Each notebook is a lesson:

1. Read the explanation cells.
2. Run every code cell in order with **Shift + Enter** (play with them — change things and re-run!).
3. At the **Exercise** cells, replace the `None  # TODO` with your answer, then run the **Check** cell below it.
4. Stuck? Every lesson has a fully-worked twin in the `solutions/` folder.

| # | Lesson |
|---|---|
| 00 | Welcome & first steps (you are here) |
| 01 | DataFrame basics |
| 02 | Reading & writing data |
| 03 | Columns & built-in functions |
| 04 | Aggregations |
| 05 | Joins |
| 06 | Window functions |
| 07 | Spark SQL |
| 08 | Complex types: structs, arrays, maps, JSON |
| 09 | UDFs & pandas UDFs |
| 10 | Under the hood: partitions, shuffles, plans, RDDs |
| 11 | Performance tuning |
| 12 | Capstone project |
| 13 | Bonus: Structured Streaming |

## What problem does Spark solve?

Tools like **pandas** load all your data into the memory of *one* computer. That's great until the data
is 500 GB and your laptop has 16 GB of RAM.

**Apache Spark** is an engine for processing data that is too big (or too slow) for one machine. It:

* splits the data into chunks called **partitions**,
* sends those chunks to many worker processes (**executors**), possibly on many machines (a **cluster**),
* processes all the chunks **in parallel**, and combines the results.

```
            ┌──────────────── Driver (your Python program) ────────────────┐
            │  turns your code into a plan, splits the work into tasks      │
            └────────┬───────────────────┬───────────────────┬──────────────┘
                     ▼                   ▼                   ▼
                Executor 1          Executor 2          Executor 3
              [partition 0,1]     [partition 2,3]     [partition 4,5]
```

The beautiful part: **the same code** runs on your laptop or on a 1,000-machine cluster.
In this course we use **local mode** — the driver and executors all live on your laptop, and each CPU core
acts like a mini-executor. Everything you learn transfers directly to Databricks, AWS EMR, GCP Dataproc, etc.

### Why did we need to install Java?
Spark itself is written in Scala and runs on the Java Virtual Machine (JVM). **PySpark** is the Python
API: your Python code sends instructions to the JVM, which does the heavy lifting. That's why the
heavy work is fast even though you write Python.

## Step 1 · Create a `SparkSession`

The `SparkSession` is your entry point to everything in Spark. You create it once per program/notebook.
`getOrCreate()` means "give me the existing one if there is one" — so re-running the cell is harmless.

Every lesson starts with this same setup cell. Run it now (the first run takes a few seconds while Java starts).

In [1]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [2]:
spark   # the session object — notice the Spark version

## Step 2 · Your first DataFrame

A **DataFrame** is a table: rows and named, typed columns — like a pandas DataFrame or a SQL table.
The difference is that a Spark DataFrame is *distributed*: its rows are spread across partitions.

`spark.range(n)` is the quickest way to make one: a single column called `id` holding 0 … n-1.

In [3]:
df = spark.range(10)
df.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



In [4]:
df.printSchema()   # column names and types. "long" = 64-bit integer

root
 |-- id: long (nullable = false)



## Step 3 · Lazy evaluation — *the* most important idea in Spark

Spark operations come in two flavours:

| Kind | What it does | Examples |
|---|---|---|
| **Transformation** | Describes a *new* DataFrame. Nothing is computed yet — Spark just writes down the recipe. | `select`, `filter`, `withColumn`, `groupBy`, `join`, `orderBy` |
| **Action** | Asks for a *result*. This is when Spark actually runs the whole recipe. | `show`, `count`, `collect`, `take`, `first`, `write` |

Watch the timings:

In [5]:
import time

big = spark.range(0, 50_000_000)                     # 50 million rows

t = time.time()
evens = big.filter(big.id % 2 == 0)                  # transformation
squared = evens.withColumn("sq", evens.id * evens.id)  # transformation
print(f"Building the recipe took {time.time() - t:.3f}s  (nothing ran!)")

t = time.time()
print("Row count:", squared.count())                 # action → now Spark works
print(f"Running it took {time.time() - t:.3f}s")

Building the recipe took 0.039s  (nothing ran!)


Row count: 25000000
Running it took 0.271s


**Why be lazy?** Because Spark sees your *whole* recipe before running it, its optimizer (called
*Catalyst*) can rearrange it: skip columns you never use, push filters earlier, pick the best join strategy…

You can see the plan Spark built with `.explain()` (we'll learn to read these properly in lesson 10):

In [6]:
squared.explain()

== Physical Plan ==
*(1) Project [id#5L, (id#5L * id#5L) AS sq#9L]
+- *(1) Filter ((id#5L % 2) = 0)
   +- *(1) Range (0, 50000000, step=1, splits=10)




## Step 4 · The Spark UI

While your session is running, Spark serves a web dashboard showing every job, how long each step took,
and how much data moved around. Open the URL printed below in your browser and click the **Jobs** tab —
you'll see the `count` you just ran.

In [7]:
print(spark.sparkContext.uiWebUrl)

http://localhost:4040


## Spark vs pandas, at a glance

| | pandas | PySpark |
|---|---|---|
| Where data lives | memory of one machine | partitions spread over a cluster |
| Execution | eager (runs immediately) | lazy (runs on an action) |
| Mutability | you can modify a DataFrame in place | DataFrames are **immutable** — every operation returns a new one |
| Row order | meaningful (has an index) | **no guaranteed order** unless you `orderBy` |
| Best for | data up to a few GB | GBs → PBs |

### Exercise 1
Create a DataFrame called `nums` containing the numbers **1 to 100 inclusive**.

*Hint:* `spark.range(start, end)` — the end is *exclusive*, like Python's `range`.

In [8]:
nums = spark.range(1, 101)

In [9]:
# Check — run me after your answer
assert nums is not None, "nums is still None"
assert nums.count() == 100, "should have 100 rows"
assert nums.agg(F.min("id"), F.max("id")).first() == (1, 100), "should start at 1 and end at 100"
print("Correct!")

Correct!


### Exercise 2
Using `nums`, store the **number of odd numbers** in a Python variable `odd_count`.

*Hint:* copy the `filter` pattern from Step 3 and finish with the `.count()` action.

In [10]:
odd_count = nums.filter(nums.id % 2 == 1).count()

In [11]:
# Check — run me after your answer
assert odd_count == 50, f"expected 50, got {odd_count}"
print("Correct!")

Correct!


### Exercise 3
Which of these are **actions**? Put their names in a Python set called `actions`.

`"filter"`, `"count"`, `"withColumn"`, `"show"`, `"select"`, `"collect"`

In [12]:
actions = {"count", "show", "collect"}

In [13]:
# Check — run me after your answer
assert actions == {"count", "show", "collect"}, "not quite — actions are the ones that produce a result"
print("Correct!")

Correct!


## Recap
* Spark processes data in **partitions**, in **parallel**. The **driver** plans, **executors** do the work.
* `SparkSession` is the entry point. A **DataFrame** is a distributed table.
* **Transformations** are lazy recipes; **actions** trigger execution.